# 05 — Benchmark Efisiensi (RTX 4050 lokal 6 GB)

Notebook ini dijalankan **setelah seluruh training selesai**. Komputer harus tersambung ke daya listrik dengan mode daya tetap, dan tidak boleh ada proses lain yang memakai GPU.

Untuk setiap model final yang sudah ada, diukur:
- jumlah parameter (total dan yang dilatih), ukuran model, dan GFLOPs per window;
- latensi per window dengan batch 1 (median dan P95), throughput, dan memori puncak, di GPU 4050 dan di CPU lokal;
- biaya training: detik per epoch dan total waktu dari `result.json`, serta VRAM puncak training.

Latensi hanya mencakup inferensi model. I/O, pra-pemrosesan sinyal, rendering citra, dan ekstraksi fitur tidak termasuk.

**Versi R15.** Angka resmi berasal dari Colab L4 (satu akun per backbone), digabung di `outputs_colab_r15`. Benchmark memakai model final hasil Colab; model utama paper adalah **model lengkap**: CNN + Beat-Level Specialist + fusi regresi logistik + kunci VT. Hasil CNN saja dilaporkan sebagai komponen (ablasi).


In [ ]:
import os
from pathlib import Path as _P
_folder = next(p for p in [_P.cwd(), *_P.cwd().parents] if (p / "v4_pipeline.py").exists() or (p / "eksperimen_v4_svdb" / "v4_pipeline.py").exists())
_folder = _folder if (_folder / "v4_pipeline.py").exists() else _folder / "eksperimen_v4_svdb"
os.environ["V4_OUTPUT_ROOT"] = str(_folder / "outputs_colab_r15")
print("Hasil dibaca dari:", os.environ["V4_OUTPUT_ROOT"])
import sys, json, time
from pathlib import Path
FOLDER = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "v4_pipeline.py").exists() or (p / "eksperimen_v4_svdb" / "v4_pipeline.py").exists())
FOLDER = FOLDER if (FOLDER / "v4_pipeline.py").exists() else FOLDER / "eksperimen_v4_svdb"
sys.path.insert(0, str(FOLDER))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
import v4_runner as R
import v4_figures as F
print("Folder eksperimen:", FOLDER)

import v4_benchmark as B
R.status()


In [ ]:
rows = []
for arch in R.ARCHS:
    for scenario in R.SCENARIOS:
        if (R.protocol().out / "final_models" / arch / scenario / "final" / "result.json").exists():
            print(time.strftime("%H:%M"), arch, scenario, flush=True)
            rows.append(B.benchmark(arch, scenario))
pd.DataFrame(rows)


In [ ]:
display(F.efficiency_table()); F.fig_tradeoff(); plt.show()
